In [1]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import json
from collections import defaultdict

import scienceplots

In [2]:
DARK_BLUE  = (0.0, 0.4667, 0.7333)
LIGHT_BLUE = (0.2, 0.7333, 0.9333)
GREEN      = (0.0, 0.6, 0.5333)
ORANGE     = (0.9333, 0.4667, 0.2)
RED        = (0.8, 0.2, 0.0667)
PINK       = (0.9333, 0.2, 0.4667)
GRAY       = (0.7333, 0.7333, 0.7333)

SAVE = False

In [ ]:
FOLDER = '/Users/juliuswuerzler/Desktop/2Pong/New/New Pong/THESIS/New/FB'
N_TRIALS = 80
T_INIT = 300
DT = 0.1
DT_RECORD = 10

data = {}

for trial in range(N_TRIALS):
    data[trial] = {}
    
    for file in os.listdir(FOLDER):
        file_name = os.path.join(FOLDER, file)
        
        if f"_{trial}_initial_weights.csv" in file:
            loaded_data = np.genfromtxt(file_name, delimiter=',', names=True, dtype=None, encoding='utf-8')
            data[trial]['ws_init'] = loaded_data
        if f"_{trial}_final_weights.csv" in file:
            loaded_data = np.genfromtxt(file_name, delimiter=',', names=True, dtype=None, encoding='utf-8')
            data[trial]['ws_final'] = loaded_data
            
        if f"_{trial}_neuron_ids" in file:
            with open(file_name, "r") as f:
                neuron_ids = json.load(f)
                data[trial]['ids'] = neuron_ids
                
        if f"_{trial}_results.csv" in file:
            with open(file_name, "r") as f:
                results = np.loadtxt(f)
                data[trial]['s_acc'] = np.mean(results[:50])
                data[trial]['e_acc'] = np.mean(results[-50:])

In [ ]:
def collect_weights(data, n_trials, mask_fn=None):
    start, end = [], []
    for i in range(n_trials):
        mask = mask_fn(data[i]) if mask_fn else slice(None)
        start.append(data[i]['ws_init']['w'][mask])
        end.append(data[i]['ws_final']['w'][mask])
    return np.concatenate(start), np.concatenate(end)

def stim_mask(trial):
    stim_ids = trial['ids']['stimulation_ids']
    return np.isin(trial['ws_init']['pre'], stim_ids) | np.isin(trial['ws_init']['post'], stim_ids)

all_start_weights,      all_end_weights      = collect_weights(data, N_TRIALS)
stim_start_weights,     stim_end_weights     = collect_weights(data, N_TRIALS, stim_mask)
not_stim_start_weights, not_stim_end_weights = collect_weights(data, N_TRIALS, lambda t: ~stim_mask(t))

In [ ]:
plt.style.use(['nature_d'])
fig, axs = plt.subplot_mosaic([["Rates", "All", "Diff"]], sharey=False)

bins = np.arange(0, 0.5001, 0.02)
axs["All"].hist(all_start_weights, bins=bins, alpha=0.5, label='start', color=DARK_BLUE)
axs["All"].hist(all_end_weights, bins=bins,alpha=0.5, label='end', color=ORANGE)
axs['All'].set_yscale('log')

bins = np.arange(0, 0.5001, 0.02)
axs["Rates"].hist(stim_end_weights, bins=bins, alpha=0.5, label='start', color=DARK_BLUE)
axs["Rates"].hist(not_stim_end_weights, bins=bins,alpha=0.5, label='end', color=ORANGE)
axs['Rates'].set_yscale('log')

bins = np.arange(-0.15, 0.34001, 0.02)
axs["Diff"].hist(stim_end_weights - stim_start_weights, bins=bins, alpha=0.5, label='stimulated', color=LIGHT_BLUE)
axs["Diff"].hist(not_stim_end_weights - not_stim_start_weights, bins=bins, alpha=0.5, label='rest', color=GRAY)
axs['Diff'].set_yscale('log')

plt.show()

In [6]:
K_PER_ELECTRODE = 3
N_ELECTRODES = 8
stim_regions = [x for x in range(N_ELECTRODES)]
post_regions = ["U", "D", "R"]
phases = ["init", "final"]


for i in range(N_TRIALS):
    d = data[i]
    d['plot_data'] = {}
    
    pre_ids = d['ws_init']['pre']
    post_ids = d['ws_init']['post']
    
    for stim_region in stim_regions:
        d['plot_data'][stim_region] = {}
        
        for post_region in post_regions:
            d['plot_data'][stim_region][post_region] = {}
            
            pre_mask = np.isin(pre_ids, d['ids']['stimulation_ids'][stim_region*K_PER_ELECTRODE:(stim_region+1)*K_PER_ELECTRODE])

            post_mask = np.array([])
            if post_region != "R":
                post_mask = np.isin(post_ids, d['ids'][f'motor_ids_{post_region}'])
            else:
                post_mask = ~(np.isin(post_ids, d['ids'][f'motor_ids_U']) | np.isin(post_ids, d['ids'][f'motor_ids_D']))
            
            for phase in phases:
                d['plot_data'][stim_region][post_region][phase] = d[f'ws_{phase}']['w'][pre_mask & post_mask]
                

# collect data for correlation calculation
corr_data = np.zeros((N_TRIALS, N_ELECTRODES, 4))
for j in range(N_TRIALS):
    for i in range(N_ELECTRODES):
        d = data[j]
        dp = d['plot_data']
        
        improvement = d['e_acc'] - d['s_acc']
        
        for k, post_region in enumerate(post_regions):
            w_change = dp[i][post_region]['final'] - dp[i][post_region]['init']
            if w_change.size < 1:
                w_change = np.zeros(1)
            
            corr_data[j][i][0] = improvement
            corr_data[j][i][k+1] = w_change.mean()

KeyError: 'ids'

In [ ]:
plt.style.use(['nature_d'])
fig, axs = plt.subplots(2, N_ELECTRODES // 2, sharex=True, sharey=True)

for i, ax in enumerate(axs.flatten()):
    
    for j, post_region in enumerate(post_regions):
        
        x = corr_data[:, -i-1, 0]
        y = corr_data[:, -i-1, j+1]
        
        # Set color
        if post_region == "U":
            color = LIGHT_BLUE
        elif post_region == "D":
            color = PINK
        else:
            color = GRAY
        
        # Scatter
        ax.scatter(x, y, alpha=0.1, color=color)
        
        # Correlation
        r = np.corrcoef(x, y)[0, 1]
        
        ax.text(
            0.05, 0.9 - j * 0.1,
            f"{post_region}: r = {r:.2f}",
            transform=ax.transAxes,
            verticalalignment='top',
            color=color
        )
        
        # ----- Regression line -----
        slope, intercept = np.polyfit(x, y, 1)
        x_fit = np.linspace(x.min(), x.max(), 100)
        y_fit = slope * x_fit + intercept
        ax.plot(x_fit, y_fit, color=color, linewidth=2)

w, h = fig.get_size_inches()
fig.set_size_inches(w, 4.5)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import pearsonr

plt.style.use(['nature_d'])
fig, axs = plt.subplots(2, N_ELECTRODES // 2, sharex=True, sharey=True)

for i, ax in enumerate(axs.flatten()):
    
    for j, (color, d) in enumerate(zip([PINK, GRAY],
                                       [corr_data[:,-i-1, 1] - corr_data[:,-i-1, 2],
                                        corr_data[:,-i-1,3]])):
        
        x = corr_data[:, -i-1, 0]
        y = d
        ax.scatter(x, y, alpha=0.1, color=color)
        
        # Correlation + significance
        r, p = pearsonr(x, y)
        
        # Optional significance stars
        if p < 0.001:
            sig = "***"
        elif p < 0.01:
            sig = "**"
        elif p < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        
        ax.text(
            0.05, 0.9 - j * 0.1,
            f"r = {r:.2f} {sig}",
            transform=ax.transAxes,
            verticalalignment='top',
            color=color
        )
        
        # Regression line
        slope, intercept = np.polyfit(x, y, 1)
        x_fit = np.linspace(x.min(), x.max(), 100)
        y_fit = slope * x_fit + intercept
        ax.plot(x_fit, y_fit, color=color, linewidth=2)

w, h = fig.get_size_inches()
fig.set_size_inches(w, 4.5)

plt.tight_layout()
plt.show()

In [ ]:
plt.style.use(['nature_d'])
fig, axs = plt.subplots(2, N_ELECTRODES // 2, sharex=True, sharey=True)

correlations = []

for i, ax in enumerate(axs.flatten()):
    x = corr_data[:, i, 0]
    y = corr_data[:, i, 1] - corr_data[:, i, 2]
    
    ax.scatter(x, y, color=LIGHT_BLUE, alpha=0.5)
    # Pearson correlation
    r, p = pearsonr(x, y)
    
    # Optional significance stars
    if p < 0.001:
        sig = "***"
    elif p < 0.01:
        sig = "**"
    elif p < 0.05:
        sig = "*"
    else:
        sig = "n.s."
    
    ax.text(
        0.05, 0.9, 
        f"r = {r:.2f} - {sig}", 
        transform=ax.transAxes,
        verticalalignment='top'
    )
    
    # ----- Regression line -----
    slope, intercept = np.polyfit(x, y, 1)
    x_fit = np.linspace(x.min(), x.max(), 100)
    y_fit = slope * x_fit + intercept
    ax.plot(x_fit, y_fit, color=color, linewidth=2)

plt.tight_layout()

In [ ]:
stim_regions = ["U", "D"]
post_regions = ["U", "D", "R"]
phases = ["init", "final"]

# weights[stim_region][stim_chunk_idx][post_region][phase] = list of arrays
weights = {
    stim: {}  # we'll create chunk entries dynamically per trial
    for stim in stim_regions
}

for stim_region in stim_regions:
    for i in range(N_TRIALS):
        d = data[i]

        pre_ids = d["ws_init"]["pre"]
        post_ids = d["ws_init"]["post"]

        # pick which side you want to scan in chunks of 3:
        # - for "D": take from the start
        # - for "U": take from the end
        stim_all = d["ids"]["stimulation_ids"]
        if stim_region == "D":
            stim_seq = stim_all
        else:
            stim_seq = stim_all[::-1]  # reverse so "end" becomes "start" for chunking

        # post masks (same for all chunks)
        post_u_mask = np.isin(post_ids, d["ids"]["motor_ids_U"])
        post_d_mask = np.isin(post_ids, d["ids"]["motor_ids_D"])
        post_r_mask = ~(post_u_mask | post_d_mask)

        # ensure per-trial storage exists
        d.setdefault("sel_weights", {})
        d["sel_weights"].setdefault(stim_region, {})  # will fill by chunk

        # iterate stimulation ids in steps of 3
        for chunk_idx, start in enumerate(range(0, len(stim_seq), 3)):
            stim_ids_chunk = stim_seq[start:start + 3]
            if len(stim_ids_chunk) == 0:
                continue

            pre_mask = np.isin(pre_ids, stim_ids_chunk)

            # ensure per-trial per-chunk structure
            d["sel_weights"][stim_region].setdefault(chunk_idx, {post: {} for post in post_regions})

            # ensure global per-chunk structure
            weights[stim_region].setdefault(
                chunk_idx,
                {post: {phase: [] for phase in phases} for post in post_regions}
            )

            for phase in phases:
                w = d[f"ws_{phase}"]["w"]

                wU = w[pre_mask & post_u_mask]
                wD = w[pre_mask & post_d_mask]
                wR = w[pre_mask & post_r_mask]

                # 1) store into data dict (per trial, per chunk)
                d["sel_weights"][stim_region][chunk_idx]["U"][phase] = wU
                d["sel_weights"][stim_region][chunk_idx]["D"][phase] = wD
                d["sel_weights"][stim_region][chunk_idx]["R"][phase] = wR

                # 2) also collect globally (per chunk)
                weights[stim_region][chunk_idx]["U"][phase].append(wU)
                weights[stim_region][chunk_idx]["D"][phase].append(wD)
                weights[stim_region][chunk_idx]["R"][phase].append(wR)

# concatenate at the end (guard empties)
for stim in stim_regions:
    for chunk_idx in weights[stim]:
        for post in post_regions:
            for phase in phases:
                lst = weights[stim][chunk_idx][post][phase]
                weights[stim][chunk_idx][post][phase] = (
                    np.concatenate(lst) if len(lst) else np.array([])
                )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from scipy.stats import pearsonr
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

def nanmean_safe(vals):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    return np.nan if vals.size == 0 else float(vals.mean())

def mean_change(arr_init, arr_final):
    """Mean of (final - init) for selected weights."""
    if arr_init is None or arr_final is None:
        return np.nan
    a = np.asarray(arr_init)
    b = np.asarray(arr_final)
    if a.size == 0 or b.size == 0:
        return np.nan
    m = min(a.size, b.size)
    return float(np.nanmean(b[:m] - a[:m]))

# --- performance per trial ---
perf = np.full(N_TRIALS, np.nan, dtype=float)
for t in range(N_TRIALS):
    d = data[t]
    perf[t] = d["e_acc"] - d["s_acc"]

# --- find chunks that exist in BOTH stim regions ("U" and "D") across trials ---
all_chunks_U = set()
all_chunks_D = set()

for t in range(N_TRIALS):
    d = data[t]
    if "sel_weights" not in d:
        continue
    if "U" in d["sel_weights"]:
        all_chunks_U |= set(d["sel_weights"]["U"].keys())
    if "D" in d["sel_weights"]:
        all_chunks_D |= set(d["sel_weights"]["D"].keys())

chunk_ids = sorted(all_chunks_U & all_chunks_D)

# --- compute correlations per chunk ---
r_by_chunk, p_by_chunk, n_by_chunk = [], [], []

for c in chunk_ids:
    delta_diff = np.full(N_TRIALS, np.nan, dtype=float)

    for t in range(N_TRIALS):
        d = data[t]
        try:
            swU = d["sel_weights"]["U"][c]  # stim_region U, this chunk
            swD = d["sel_weights"]["D"][c]  # stim_region D, this chunk
        except KeyError:
            continue

        # correct pathways: U->U and D->D
        dUU = mean_change(swU["U"]["init"], swU["U"]["final"])
        dDD = mean_change(swD["D"]["init"], swD["D"]["final"])

        # wrong pathways: U->D and D->U
        dUD = mean_change(swU["D"]["init"], swU["D"]["final"])
        dDU = mean_change(swD["U"]["init"], swD["U"]["final"])

        delta_correct = nanmean_safe([dUU, dDD])
        delta_wrong   = nanmean_safe([dUD, dDU])

        delta_diff[t] = delta_correct - delta_wrong

    mask = np.isfinite(delta_diff) & np.isfinite(perf)
    n = int(mask.sum())
    n_by_chunk.append(n)

    if n < 2:
        r_by_chunk.append(np.nan)
        p_by_chunk.append(np.nan)
    else:
        x = delta_diff[mask]
        y = perf[mask]
        if _HAS_SCIPY:
            r, p = pearsonr(x, y)
        else:
            r = float(np.corrcoef(x, y)[0, 1])
            p = np.nan
        r_by_chunk.append(r)
        p_by_chunk.append(p)

# --- plot r vs chunk ---
plt.figure()
plt.plot(chunk_ids, r_by_chunk, marker="o")
plt.axhline(0, linewidth=1)
plt.xlabel("Stimulation chunk index")
plt.ylabel("Pearson r: corr(Δcorrect-Δwrong, performance)")
plt.title("Correlation by stimulation chunk")
plt.show()

# --- optional printout ---
print("chunk\tN\tr\tp")
for c, n, r, p in zip(chunk_ids, n_by_chunk, r_by_chunk, p_by_chunk):
    if _HAS_SCIPY:
        print(f"{c}\t{n}\t{r:.4f}\t{p:.3g}")
    else:
        print(f"{c}\t{n}\t{r:.4f}\tNA")

In [ ]:
plt.style.use(['nature_d'])

fig, axs = plt.subplot_mosaic(
    [['UU', 'UD', 'UR'],
     ['DU', 'DD', 'DR']],
    sharex=True,
    sharey=True
)

for stim in stim_regions:
    for post in post_regions:
        ax = axs[f"{stim}{post}"]
        ax.set_title(f"{stim}→{post}")
        
        color = 'black'
        if stim == post:
            color = GREEN
        elif post == "R":
            color = GRAY
        else:
            color = RED
            
        ax.hist(weights[stim][post]["final"] - weights[stim][post]["init"], label="final", density=True, color=color)
        
        mean_change = np.mean(weights[stim][post]["final"] - weights[stim][post]["init"])
        ax.axvline(mean_change,  color='black', linestyle='--', label='mean', linewidth=1.5)
        print(f"{stim}->{post}: {mean_change:.4f}")
        
        if post == "U":
            ax.set_ylabel('density')
            ax.set_ylabel('density')
        
fig.text(0.5, 0.02, r'$\Delta$w', 
         ha='center', va='center')

w, h = fig.get_size_inches()
fig.set_size_inches(w, 4) 
    